# Seeing Clusters: 1D, 2D, and 3D

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/unsupervised/02-clustering/clusters_1d_2d_3d.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/unsupervised/02-clustering"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Learning Objectives

- Connect clustering to familiar data analysis: distributions and summary statistics.
- **See** natural groupings in 1D, 2D, and 3D before studying specific algorithms.
- Use plots (KDE, scatter, 3D scatter, faceted panels) to make groups visible.
- Summarize each group with per-cluster statistics.
- **Interpret** clusters in domain language — what each group means for the business or problem, not just the numbers.
- Distinguish adding a **numerical** vs **categorical** third variable.


## Introduction

Clustering is about finding **groups of similar observations** in data — segments, tiers, or blobs that stand out when you look closely.

This notebook focuses on the **what**: learning to recognize those groups visually, summarize them with statistics you already use in exploratory analysis, and **name what they mean** in the language of the data’s domain (spend tiers, shopper personas, and so on). We deliberately defer the **how** — choosing *k*, measuring cluster quality, and understanding optimization — to later notebooks.

We build up from one variable, to two, to three, using tools you already know (distribution plots and `describe()`) before any algorithm becomes the main topic.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

In [ ]:
# show 3 head and 3 tail rows
pd.set_option("display.min_rows", 6)

## 1D

Start with a **single numeric variable**. In everyday data analysis you would inspect its **distribution** — shape, spread, and whether values pile up in separate regions — and summarize it with statistics like count, mean, and standard deviation.

Here we use a toy example: customer **annual spend** with three obvious tiers on a number line. A KDE plot with rug marks plays a similar role to a histogram: it shows where values concentrate. We run K-Means with *K* = 3 only to **assign consistent colors** to each tier for the plots below — not because we are learning the algorithm yet.


In [ ]:
# 1D array of customer annual spend ($)
spend = np.array([15, 22, 19, 140, 150, 165, 890, 920, 950]).reshape(-1, 1)

# Fit K-Means with K=3 tiers
kmeans = KMeans(n_clusters=3, random_state=42).fit(spend)


Each point sits on the spend axis, colored by its assigned group. Dashed vertical lines mark the group centers — the typical spend level within each tier.


In [ ]:
# Visualize 1D spend points + K-Means centers
centers = sorted(kmeans.cluster_centers_.flatten())

fig, ax = plt.subplots(figsize=(9, 2.8))
sns.stripplot(
    x=spend.flatten(),
    hue=kmeans.labels_,
    palette="deep",
    size=12,
    jitter=0.2,
    ax=ax,
)
for c in centers:
    ax.axvline(c, color="black", linestyle="--", linewidth=1.5, alpha=0.8)
    ax.text(c, 0.55, f"{c:.1f}", ha="center", va="bottom", fontsize=10)

ax.set_xlabel("Annual spend ($)")
ax.set_yticks([])
ax.set_title("K-Means tiers (K=3): data points + cluster centers")
ax.legend(title="Cluster", loc="upper center")
sns.despine(left=True)
plt.tight_layout()
plt.show()


Within each group, the KDE curve shows how values are spread — like inspecting a separate histogram for every segment. Rug ticks mark the individual observations.


In [ ]:
# Kernel density estimate of spend within each cluster (same figure)
plot_df = pd.DataFrame({"spend": spend.flatten(), "cluster": kmeans.labels_})
palette = sns.color_palette("deep", 3)

fig, ax = plt.subplots(figsize=(9, 4))
for cluster_id in sorted(plot_df["cluster"].unique()):
    subset = plot_df.loc[plot_df["cluster"] == cluster_id, "spend"]
    color = palette[cluster_id]

    sns.kdeplot(
        subset,
        ax=ax,
        color=color,
        fill=True,
        alpha=0.25,
        linewidth=2,
        label=f"Cluster {cluster_id}",
    )
    sns.rugplot(subset, ax=ax, color=color, height=0.04)
    center = kmeans.cluster_centers_[cluster_id, 0]
    ax.axvline(center, color=color, linestyle="--", linewidth=1.5, alpha=0.9)

ax.set_xlabel("Annual spend ($)")
ax.set_ylabel("Density")
ax.set_title("Spend distribution within each K-Means cluster (KDE)")
ax.legend(title="Cluster")
sns.despine()
plt.tight_layout()
plt.show()

### Cluster Statistics

Numbers complement the pictures. A per-cluster `describe()` table is how you **characterize** each group: how many points it contains, its typical value, and how spread out it is.


In [ ]:
pd.DataFrame({"spend": spend.flatten(), "cluster": kmeans.labels_}).groupby(
    "cluster"
)["spend"].describe()

### What do these clusters describe?

In a **customer spend** setting, the three groups are spending tiers — not abstract blobs on a line:

| Cluster | Typical spend | Domain reading |
|--------:|--------------:|----------------|
| 2 | ~$19 | **Budget / low-activity** customers — minimal annual spend. |
| 0 | ~$152 | **Regular mid-tier** customers — moderate, steady spenders. |
| 1 | ~$920 | **High-value / VIP** customers — orders of magnitude more spend than the rest. |

A retailer would treat these differently: retention offers for mid-tier shoppers, white-glove service or loyalty perks for the VIP tier, and activation campaigns for the low-spend group. The algorithm only partitioned the number line; the **business meaning** comes from reading the centers against the spend domain.


---

## 2D

The same idea extends to **two features at once**. Instead of a number line or density curve, you use a **scatter plot**: each point is positioned by two coordinates, and groups appear as visible blobs on the plane.

We use the [**Mall Customers**](../data/Mall_Customers.csv) dataset — 200 mall shoppers with Age, Genre, Annual Income, and Spending Score. For now we cluster on **Income** and **Spending Score** only; Age and Genre are held back for the 3D section.


In [ ]:
df = pd.read_csv("../data/Mall_Customers.csv")
df

We set *K* = 5 because five distinct blobs are visible in this dataset — a choice made for visualization, not via the elbow method or silhouette score (those come later). Black **X** marks show cluster centers; the summary table below reports mean income and spending score for each group.


In [ ]:
# Two features → shape (200, 2). A DataFrame is already 2D, so no reshape.
X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

kmeans_2d = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X)
labels = kmeans_2d.labels_
centers = kmeans_2d.cluster_centers_

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue=labels,
    palette="deep",
    s=60,
    ax=ax,
)
ax.scatter(
    centers[:, 0],
    centers[:, 1],
    c="black",
    s=200,
    marker="X",
    label="centers",
    zorder=5,
)
ax.set_title("Mall customers: K-Means with K=5")
ax.legend(title="Cluster")
sns.despine()
plt.tight_layout()
plt.show()

df.assign(cluster=labels).groupby("cluster")[
    ["Annual Income (k$)", "Spending Score (1-100)"]
].describe()


### What do these clusters describe?

On the mall’s **Income × Spending Score** plane, the five blobs are shopper **personas** — how much people earn vs. how freely they spend at the mall:

| Cluster | Income | Spending score | Domain reading |
|--------:|-------:|---------------:|----------------|
| 4 | low (~$26k) | low (~21) | **Price-sensitive / low engagement** — limited income and limited mall activity. |
| 2 | low (~$26k) | high (~79) | **Enthusiastic budget shoppers** — spend a lot relative to income (often younger). |
| 0 | mid (~$55k) | mid (~50) | **Mainstream / average shoppers** — the largest, “typical” segment. |
| 3 | high (~$88k) | low (~17) | **Affluent but cautious** — high earners who barely use the mall. |
| 1 | high (~$87k) | high (~82) | **High-value targets** — both the means and the habit to spend heavily. |

Marketing implication: Cluster 1 is the prime loyalty audience; Cluster 3 is a conversion opportunity (income without spend); Cluster 2 may respond to promotions despite low income; Cluster 4 needs different messaging than VIPs. Again — K-Means found the blobs; **naming the personas** is the domain step.


---

## 3D

Real datasets rarely stop at two columns. There are two practical ways to bring in a **third variable**:

1. **Numerical** third feature → add a literal third axis to the plot.
2. **Categorical** third feature → keep the same 2D plane and split it into side-by-side panels, one per category.


### Numerical 3rd Variable

We add **Age** as a third numeric axis alongside Income and Spending Score. The five income × score blobs remain the main structure; Age often softens or stretches them rather than creating entirely new groups — but it is worth seeing in 3D.

The scatter plot includes orthogonal guide lines so you can read each point's coordinates. Per-cluster statistics now cover all three numeric features. Note that clustering in full 3D numeric space can **reshape** groups compared to the 2D view — compare this section mentally with the previous one.


In [ ]:
# --- Numerical 3rd axis: Age ---
# Does Age "belong"? The five income×score blobs are still the main structure;
# Age often softens them rather than inventing brand-new ones. Still useful to see.
X3 = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]]
kmeans_3d = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X3)
labels_3d = kmeans_3d.labels_
centers_3d = kmeans_3d.cluster_centers_


def drop_lines_to_planes(ax, x, y, z, color="black", alpha=0.55, lw=1.2):
    """Orthogonal dotted guides from axis planes to a 3D point."""
    xmin, _ = ax.get_xlim()
    ymin, _ = ax.get_ylim()
    zmin, _ = ax.get_zlim()
    # parallel to Age (x): from yz-plane
    ax.plot([xmin, x], [y, y], [z, z], color=color, linestyle=":", linewidth=lw, alpha=alpha)
    # parallel to Income (y): from xz-plane
    ax.plot([x, x], [ymin, y], [z, z], color=color, linestyle=":", linewidth=lw, alpha=alpha)
    # parallel to Spending (z): from xy-plane
    ax.plot([x, x], [y, y], [zmin, z], color=color, linestyle=":", linewidth=lw, alpha=alpha)


fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    X3["Age"],
    X3["Annual Income (k$)"],
    X3["Spending Score (1-100)"],
    c=labels_3d,
    cmap="tab10",
    s=40,
    alpha=0.85,
)
ax.scatter(
    centers_3d[:, 0],
    centers_3d[:, 1],
    centers_3d[:, 2],
    c="black",
    s=180,
    marker="X",
    depthshade=False,
    label="centers",
    zorder=5,
)
for cx, cy, cz in centers_3d:
    drop_lines_to_planes(ax, cx, cy, cz)

ax.set_xlabel("Age")
ax.set_ylabel("Annual Income (k$)")
ax.set_zlabel("Spending Score (1-100)")
ax.set_title("3D via a numerical 3rd feature (Age)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

df.assign(cluster=labels_3d).groupby("cluster")[
    ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
].describe()


### What do these clusters describe?

Adding **Age** keeps the same income/spend story but layers in a life-stage reading of each persona:

| Cluster | Age | Income | Spending | Domain reading |
|--------:|----:|-------:|---------:|----------------|
| 4 | young (~26) | low | high | **Young high-engagement shoppers** — early-career, spend eagerly at the mall. |
| 2 | young-adult (~33) | high | high | **Prime high-value customers** — peak earning *and* spending years. |
| 1 | mid (~43) | mid | mid | **Established average shoppers** — the large middle of the mall population. |
| 0 | mid (~45) | low | low | **Older low-activity shoppers** — limited income and limited spend. |
| 3 | mid (~41) | high | low | **Affluent under-spenders** — means without mall habit (same opportunity as in 2D). |

Compared with the 2D view, Age mostly **softens and reorders** the same income×score structure rather than inventing brand-new business segments. The domain takeaway: age helps prioritize (e.g. younger vs. mid-life high-value shoppers) but income and spending score still define the core personas.


### Categorical 3rd Variable

**Genre** (Male / Female) is categorical — it cannot be a continuous axis. Instead we **facet** the same Income × Spending Score scatter into two adjacent panels, one per gender.

We reuse the **2D cluster labels** from the previous section so the panels stay directly comparable; Genre acts as "depth" without re-fitting the model. The summary table groups by both Genre and cluster, showing how each segment looks **within** each category.


In [ ]:
# --- Categorical 3rd axis: Genre (Gender) via adjacent 2D panels ---
# Same income×score plane as before; Gender is the "depth" encoded by faceting.
# Reuse the 2D cluster labels so panels stay comparable.
plot_df = df.assign(cluster=labels)

g = sns.relplot(
    data=plot_df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="cluster",
    col="Genre",
    palette="deep",
    s=60,
    height=5,
    aspect=1,
)
g.figure.suptitle(
    "3D via a categorical 3rd feature (Genre): two adjacent 2D views",
    y=1.03,
)
plt.show()

plot_df.groupby(["Genre", "cluster"])[
    ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
].describe()


### What do these clusters describe?

Faceting by **Genre** does not create new spend personas — it asks whether the same Income × Spending Score segments look different for **Female** vs **Male** shoppers.

## What's Next

You can now **recognize** clusters in low-dimensional data — see them in plots, summarize them with per-group statistics, and **translate** those groups into domain language (tiers, personas, life stages).

The next step is the **how**: choosing *k*, measuring cluster quality, and understanding how algorithms optimize assignments.
